In [ ]:
%matplotlib widget

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from scipy.signal import remez, freqz
from ipywidgets import Dropdown, FloatSlider, VBox, HBox, HTML, Layout
from IPython.display import display

# ============================================================
# DIGITAL FIR DIFFERENTIATOR
# ============================================================

plt.ioff()

CONTENT_WIDTH = '820px'

plt.rcParams.update({'font.size':10.5,'axes.titlesize':12,'axes.labelsize':10.5,'xtick.labelsize':9,'ytick.labelsize':9,'legend.fontsize':8.5})

# ============================================================
# STYLE
# ============================================================

display(HTML("""
<style>

.df-root{
    width:820px;
    max-width:820px;
    font-family:Arial,sans-serif;
}

.df-header{
    background:linear-gradient(90deg,#7b1fa2,#9c27b0);
    color:white;
    padding:11px 15px;
    border-radius:8px 8px 0 0;
    font-size:19px;
    font-weight:bold;
}

.df-doc{
    background:#fbf7fc;
    border:1px solid #d7c4e2;
    border-top:none;
    padding:11px 14px;
    border-radius:0 0 8px 8px;
    font-size:15.5px;
    line-height:1.55;
    margin-bottom:9px;
}

.df-box{
    width:100%;
    box-sizing:border-box;
    border:1px solid #d7c4e2;
    border-radius:7px;
    padding:10px 12px;
    margin-bottom:8px;
    font-size:15px;
    line-height:1.5;
}

.df-title{
    font-weight:bold;
    color:#6a1b9a;
    font-size:16px;
    margin-bottom:7px;
}

.df-cols{
    display:flex;
    gap:18px;
    align-items:flex-start;
    flex-wrap:nowrap;
}

.df-col{
    flex:1;
    min-width:0;
}

.widget-label{
    font-size:14.5px !important;
}

.jupyter-widgets input,
.jupyter-widgets select{
    font-size:14px !important;
}

.jupyter-widgets-output-area,
.widget-output,
.output_area,
.output_subarea,
.jp-OutputArea-output,
.jp-OutputArea-child{
    overflow-x:visible !important;
    overflow-y:visible !important;
    max-width:none !important;
}

.jp-OutputArea,
.output_wrapper,
.widget-box,
.jupyter-widgets{
    overflow:visible !important;
}

/* Hide the ipympl resize handle without changing canvas geometry */
.jupyter-matplotlib-canvas-div{
    resize:none !important;
}

</style>
"""))

# ============================================================
# DOCUMENTATION
# ============================================================

display(HTML("""
<div class="df-root">

<div class="df-header">
Parks–McClellan FIR Digital Differentiator
</div>

<div class="df-doc">

<b>Purpose.</b>
An ideal digital differentiator has frequency response H(e<sup>jω</sup>)=jω and therefore a magnitude proportional to frequency.
A practical FIR differentiator approximates this behavior only over a prescribed bandwidth 0≤ω≤ω<sub>p</sub>.

<br><br>

<b>Filter structure.</b>
The impulse response is antisymmetric.
An odd FIR length produces a Type-III filter, for which both H(0)=0 and H(π)=0 are forced.
An even FIR length produces a Type-IV filter, for which H(0)=0 is forced but H(π) is not.

<br><br>

<b>Minimax design.</b>
The Parks–McClellan differentiator formulation minimizes the relative error, corresponding to a frequency-dependent weighting proportional to 1/ω.

</div>

</div>
"""))

# ============================================================
# CONTROLS
# ============================================================

filter_type = Dropdown(options=['Type III','Type IV'],value='Type III',description='FIR type:',style={'description_width':'70px'},layout=Layout(width='200px'))

length_control = Dropdown(options=list(range(11,52,2)),value=21,description='Length N:',style={'description_width':'70px'},layout=Layout(width='200px'))

fp_slider = FloatSlider(value=0.80,min=0.20,max=0.95,step=0.01,description='ωp / π:',continuous_update=True,readout_format='.2f',style={'description_width':'70px'},layout=Layout(width='300px'))

controls = VBox([
    HTML('<div class="df-title">Differentiator parameters</div>'),
    HBox([filter_type,length_control,fp_slider])
],layout=Layout(width=CONTENT_WIDTH,border='1px solid #d7c4e2',padding='9px 12px',margin='0 0 8px 0'))

info = HTML(layout=Layout(width=CONTENT_WIDTH,margin='0 0 8px 0'))

# ============================================================
# CHANGE ALLOWED LENGTHS
# ============================================================

def update_length_options(change=None):
    if filter_type.value == 'Type III':
        options = list(range(11,52,2))
        default = 21
    else:
        options = list(range(12,53,2))
        default = 20

    old_value = length_control.value
    length_control.options = options

    if old_value in options:
        length_control.value = old_value
    else:
        length_control.value = default

# ============================================================
# FIGURE — CREATED ONLY ONCE
# ============================================================

fig,axes = plt.subplots(2,2,figsize=(8.2,6.2))
ax1,ax2,ax3,ax4 = axes.flat

fig.canvas.toolbar_visible = False
fig.canvas.header_visible = False
fig.canvas.footer_visible = False

# ============================================================
# MAGNITUDE RESPONSE
# ============================================================

line_mag, = ax1.plot([],[],color='red',linewidth=1.5,label='Designed differentiator')
line_desired, = ax1.plot([],[],'--',linewidth=1.0,label='Desired magnitude')
band_edge = ax1.axvline(0.80,linestyle=':',linewidth=1.0,label=r'$\omega_p$')

ax1.set_xlim(0,1)
ax1.set_ylim(0,0.55)
ax1.set_title('Differentiator Magnitude Response')
ax1.set_xlabel(r'Normalized frequency $\omega/\pi$')
ax1.set_ylabel(r'$|H(e^{j\omega})|$')
ax1.grid(True,linestyle=':',alpha=0.25)
ax1.legend(loc='upper center',bbox_to_anchor=(0.5,-0.25),ncol=2,frameon=False)

# ============================================================
# RELATIVE ERROR
# ============================================================

line_error, = ax2.plot([],[],color='red',linewidth=1.5,label='Relative error')
zero_error = ax2.axhline(0,linestyle='--',linewidth=1.0,label='Zero error')

ax2.set_xlim(0,1)
ax2.set_ylim(-0.40,0.40)
ax2.set_title('Weighted Relative Error')
ax2.set_xlabel(r'Normalized frequency $\omega/\pi$')
ax2.set_ylabel(r'$1-|H|/|H_d|$')
ax2.grid(True,linestyle=':',alpha=0.25)
ax2.legend(loc='upper center',bbox_to_anchor=(0.5,-0.25),frameon=False)

# ============================================================
# ZERO-PHASE RESPONSE
# ============================================================

line_phase, = ax3.plot([],[],color='red',linewidth=1.5,label='Zero-phase component')
zero_phase = ax3.axhline(0,linestyle='--',linewidth=1.0)

ax3.set_xlim(0,1)
ax3.set_ylim(-0.55,0.55)
ax3.set_title('Antisymmetric Zero-Phase Response')
ax3.set_xlabel(r'Normalized frequency $\omega/\pi$')
ax3.set_ylabel(r'$A(\omega)$')
ax3.grid(True,linestyle=':',alpha=0.25)
ax3.legend(loc='upper center',bbox_to_anchor=(0.5,-0.25),frameon=False)

# ============================================================
# IMPULSE RESPONSE
# ============================================================

impulse_markers, = ax4.plot([],[],'ro',markersize=3.5)

stem_collection = LineCollection([],colors='red',linewidths=1.0)
ax4.add_collection(stem_collection)

center_line = ax4.axvline(0,linestyle='--',linewidth=1.0,label='Antisymmetry center')

ax4.set_ylim(-0.23,0.23)
ax4.set_title('Antisymmetric FIR Impulse Response')
ax4.set_xlabel('Sample index $n$')
ax4.set_ylabel('$h[n]$')
ax4.grid(True,linestyle=':',alpha=0.25)
ax4.legend(loc='upper center',bbox_to_anchor=(0.5,-0.25),frameon=False)

plt.subplots_adjust(left=0.085,right=0.985,top=0.95,bottom=0.18,wspace=0.30,hspace=0.82)

# ============================================================
# UPDATE
# ============================================================

def update_differentiator(change=None):

    N = length_control.value
    fp = fp_slider.value
    ftype = filter_type.value

    try:
        h = remez(N,[0.0,fp],[1.0],type='differentiator',fs=2.0,maxiter=100,grid_density=32)

    except Exception as e:
        info.value = f'<div class="df-root"><div class="df-box"><b>Design error:</b> {e}</div></div>'
        return

    omega,H = freqz(h,worN=32768)

    fn = omega/np.pi
    mag = np.abs(H)

    mask = fn <= fp

    fn_band = fn[mask]
    mag_band = mag[mask]

    desired = 0.5*fn_band

    valid = fn_band > 1e-5

    relative_error = 1.0-mag_band[valid]/desired[valid]

    delay = (N-1)/2
    A = np.imag(H*np.exp(1j*omega*delay))

    symmetry_error = np.max(np.abs(h+h[::-1]))
    H0 = abs(np.sum(h))
    Hpi = abs(np.sum(h*(-1)**np.arange(N)))

    info.value = f"""
    <div class="df-root">

    <div class="df-box">

    <div class="df-title">Current differentiator design</div>

    <div class="df-cols">

    <div class="df-col">
    Structure: <b>{ftype}</b><br>
    FIR length: <b>N = {N}</b><br>
    Filter order: <b>{N-1}</b>
    </div>

    <div class="df-col">
    Differentiator bandwidth: <b>0 ≤ ω ≤ {fp:.2f}π</b><br>
    Desired magnitude: <b>|H<sub>d</sub>| ∝ ω</b>
    </div>

    <div class="df-col">
    Antisymmetry error: <b>{symmetry_error:.2e}</b><br>
    |H(0)|: <b>{H0:.2e}</b><br>
    |H(π)|: <b>{Hpi:.2e}</b>
    </div>

    </div>

    </div>

    </div>
    """

    # ========================================================
    # MAGNITUDE RESPONSE
    # ========================================================

    line_mag.set_data(fn,mag)
    line_desired.set_data([0,fp],[0,0.5*fp])
    band_edge.set_xdata([fp,fp])

    # ========================================================
    # RELATIVE ERROR
    # ========================================================

    line_error.set_data(fn_band[valid],relative_error)

    # ========================================================
    # ZERO-PHASE RESPONSE
    # ========================================================

    line_phase.set_data(fn,A)

    # ========================================================
    # IMPULSE RESPONSE
    # ========================================================

    n = np.arange(N)

    impulse_markers.set_data(n,h)

    segments = [np.array([[ni,0],[ni,hi]]) for ni,hi in zip(n,h)]
    stem_collection.set_segments(segments)

    center = (N-1)/2

    center_line.set_xdata([center,center])

    # Only the horizontal extent follows N.
    # The vertical scale remains fixed so coefficient changes are visible.
    ax4.set_xlim(-1,N)

    fig.canvas.draw_idle()

# ============================================================
# EVENTS
# ============================================================

def type_changed(change):
    update_length_options()
    update_differentiator()

filter_type.observe(type_changed,names='value')
length_control.observe(update_differentiator,names='value')
fp_slider.observe(update_differentiator,names='value')

# ============================================================
# DISPLAY
# ============================================================

display(controls)
display(info)
display(fig.canvas)

update_differentiator()